# 02 — PySpark Pipeline & Feature Engineering Demo

**CIS 731 Term Project — Brent Showalter**

End-to-end walkthrough of the PySpark feature engineering pipeline:
processed Parquet → feature transforms → pandas-ready training set.

**Prerequisites:** Run `spark_pipeline.py` first so processed Parquet exists at `data/processed/`.

In [1]:
import sys
from pathlib import Path

# Add project root so local packages are importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.spark_pipeline import create_spark_session
from src.features.engineering import (
    add_time_features,
    add_hunting_season_flags,
    add_lag_features,
    add_rolling_features,
    add_product_encodings,
    build_feature_set,
    to_pandas,
)
from pyspark.sql import functions as F

print("Imports OK")

Imports OK


## Load Processed Data

In [2]:
spark = create_spark_session(app_name="FeatureEngineeringDemo")

# Load processed Parquet and cast date column
df = spark.read.parquet(str(PROJECT_ROOT / "data" / "processed"))
df = df.withColumn("date", F.col("date").cast("date"))

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()
df.show(5, truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/26 16:52:46 WARN Utils: Your hostname, Brents-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.209 instead (on interface en1)
26/02/26 16:52:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/02/26 16:53:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Rows: 4,440
Columns: 9
root
 |-- subcategory: string (nullable = true)
 |-- sizing: string (nullable = true)
 |-- tactical: string (nullable = true)
 |-- date: date (nullable = true)
 |-- quantity: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- barrel_length: double (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- avg_price: double (nullable = true)



+------------+------+--------+----------+--------+---------------+------------------+-----------------+------------------+
|subcategory |sizing|tactical|date      |quantity|amount         |barrel_length     |transaction_count|avg_price         |
+------------+------+--------+----------+--------+---------------+------------------+-----------------+------------------+
|Bolt Action |12 GA |N       |2019-01-01|31      |16274.330078125|22.0              |2                |524.9783896169355 |
|Bolt Action |16 GA |N       |2019-01-01|0       |0.0            |23.23602156491863 |1                |0.0               |
|Bolt Action |20 GA |N       |2019-01-01|496     |233402.08984375|22.0              |2                |470.5687295236895 |
|Lever Action|12 GA |N       |2019-01-01|157     |48466.69140625 |22.0              |5                |308.70504080414014|
|Lever Action|12 GA |NA      |2019-01-01|0       |0.0            |19.444444444444443|1                |0.0               |
+------------+--

## Apply Feature Engineering (Step by Step)

In [3]:
# Step 1: Time features
df_time = add_time_features(df)
new_cols = [c for c in df_time.columns if c not in df.columns]
print(f"add_time_features → +{len(new_cols)} columns: {new_cols}")
df_time.select("date", *new_cols).show(5, truncate=False)

2026-02-26 16:54:27,006 INFO Added time features: month, quarter, year, cyclical, domain events


add_time_features → +10 columns: ['month', 'quarter', 'year', 'month_sin', 'month_cos', 'covid_period', 'post_covid', 'is_march', 'off_season', 'months_to_hunting']


+----------+-----+-------+----+-------------------+------------------+------------+----------+--------+----------+-----------------+
|date      |month|quarter|year|month_sin          |month_cos         |covid_period|post_covid|is_march|off_season|months_to_hunting|
+----------+-----+-------+----+-------------------+------------------+------------+----------+--------+----------+-----------------+
|2019-01-01|1    |1      |2019|0.49999999999999994|0.8660254037844387|0           |0         |0       |0         |9                |
|2019-01-01|1    |1      |2019|0.49999999999999994|0.8660254037844387|0           |0         |0       |0         |9                |
|2019-01-01|1    |1      |2019|0.49999999999999994|0.8660254037844387|0           |0         |0       |0         |9                |
|2019-01-01|1    |1      |2019|0.49999999999999994|0.8660254037844387|0           |0         |0       |0         |9                |
|2019-01-01|1    |1      |2019|0.49999999999999994|0.8660254037844387

In [4]:
# Step 2: Hunting season flags
df_hunt = add_hunting_season_flags(df_time)
new_cols = [c for c in df_hunt.columns if c not in df_time.columns]
print(f"add_hunting_season_flags → +{len(new_cols)} columns: {new_cols}")
df_hunt.select("date", "month", *new_cols).show(5, truncate=False)

2026-02-26 16:54:30,845 INFO Added hunting season flags: ['is_waterfowl_season', 'is_upland_season', 'is_turkey_spring_season', 'is_turkey_fall_season', 'is_dove_season', 'is_deer_season'] + hunting_intensity


add_hunting_season_flags → +7 columns: ['is_waterfowl_season', 'is_upland_season', 'is_turkey_spring_season', 'is_turkey_fall_season', 'is_dove_season', 'is_deer_season', 'hunting_intensity']


+----------+-----+-------------------+----------------+-----------------------+---------------------+--------------+--------------+-----------------+
|date      |month|is_waterfowl_season|is_upland_season|is_turkey_spring_season|is_turkey_fall_season|is_dove_season|is_deer_season|hunting_intensity|
+----------+-----+-------------------+----------------+-----------------------+---------------------+--------------+--------------+-----------------+
|2019-01-01|1    |1                  |1               |0                      |0                    |0             |1             |3                |
|2019-01-01|1    |1                  |1               |0                      |0                    |0             |1             |3                |
|2019-01-01|1    |1                  |1               |0                      |0                    |0             |1             |3                |
|2019-01-01|1    |1                  |1               |0                      |0                    

In [5]:
# Step 3: Lag features
df_lag = add_lag_features(df_hunt)
new_cols = [c for c in df_lag.columns if c not in df_hunt.columns]
print(f"add_lag_features → +{len(new_cols)} columns: {new_cols}")
df_lag.select("date", "subcategory", "sizing", "quantity", *new_cols).show(5, truncate=False)

2026-02-26 16:54:35,841 INFO Added lag features: lags=[1, 3, 6, 12], target=quantity, + avg_price_lag_1, yoy/mom change


add_lag_features → +7 columns: ['quantity_lag_1', 'quantity_lag_3', 'quantity_lag_6', 'quantity_lag_12', 'avg_price_lag_1', 'quantity_yoy_change', 'quantity_mom_change']


+----------+-----------+------+--------+--------------+--------------+--------------+---------------+-----------------+-------------------+-------------------+
|date      |subcategory|sizing|quantity|quantity_lag_1|quantity_lag_3|quantity_lag_6|quantity_lag_12|avg_price_lag_1  |quantity_yoy_change|quantity_mom_change|
+----------+-----------+------+--------+--------------+--------------+--------------+---------------+-----------------+-------------------+-------------------+
|2019-01-01|Bolt Action|12 GA |31      |NULL          |NULL          |NULL          |NULL           |NULL             |NULL               |NULL               |
|2019-02-01|Bolt Action|12 GA |23      |31            |NULL          |NULL          |NULL           |524.9783896169355|NULL               |-8                 |
|2019-03-01|Bolt Action|12 GA |44      |23            |NULL          |NULL          |NULL           |520.7782778532609|NULL               |21                 |
|2019-04-01|Bolt Action|12 GA |20      |

In [6]:
# Step 4: Rolling features
df_roll = add_rolling_features(df_lag)
new_cols = [c for c in df_roll.columns if c not in df_lag.columns]
print(f"add_rolling_features → +{len(new_cols)} columns: {new_cols}")
df_roll.select("date", "subcategory", "sizing", *new_cols).show(5, truncate=False)

2026-02-26 16:54:52,694 INFO Added rolling features: windows=[3, 6, 12], target=quantity


add_rolling_features → +6 columns: ['quantity_ma_3', 'quantity_std_3', 'quantity_ma_6', 'quantity_std_6', 'quantity_ma_12', 'quantity_std_12']


+----------+-----------+------+------------------+------------------+------------------+------------------+------------------+------------------+
|date      |subcategory|sizing|quantity_ma_3     |quantity_std_3    |quantity_ma_6     |quantity_std_6    |quantity_ma_12    |quantity_std_12   |
+----------+-----------+------+------------------+------------------+------------------+------------------+------------------+------------------+
|2019-01-01|Bolt Action|12 GA |31.0              |0.0               |31.0              |0.0               |31.0              |0.0               |
|2019-02-01|Bolt Action|12 GA |27.0              |5.656854249492381 |27.0              |5.656854249492381 |27.0              |5.656854249492381 |
|2019-03-01|Bolt Action|12 GA |32.666666666666664|10.598742063723098|32.666666666666664|10.598742063723098|32.666666666666664|10.598742063723098|
|2019-04-01|Bolt Action|12 GA |29.0              |13.076696830622021|29.5              |10.723805294763608|29.5             

In [7]:
# Step 5: Product encodings
df_enc = add_product_encodings(df_roll)
new_cols = [c for c in df_enc.columns if c not in df_roll.columns]
print(f"add_product_encodings → +{len(new_cols)} columns: {new_cols}")
df_enc.select("subcategory", "sizing", "tactical", *new_cols).show(5, truncate=False)

2026-02-26 16:55:15,885 INFO Added product encodings: sizing_grouped, tactical flags, subcategory one-hot (reference=Pump Action, 8 dummies), is_12ga


add_product_encodings → +12 columns: ['sizing_grouped', 'is_tactical', 'is_tactical_na', 'is_semiautomatic', 'is_single_shot', 'is_over_under', 'is_lever_action', 'is_side_by_side', 'is_bolt_action', 'is_rifle_shotgun', 'is_revolver', 'is_12ga']


+------------+------+--------+--------------+-----------+--------------+----------------+--------------+-------------+---------------+---------------+--------------+----------------+-----------+-------+
|subcategory |sizing|tactical|sizing_grouped|is_tactical|is_tactical_na|is_semiautomatic|is_single_shot|is_over_under|is_lever_action|is_side_by_side|is_bolt_action|is_rifle_shotgun|is_revolver|is_12ga|
+------------+------+--------+--------------+-----------+--------------+----------------+--------------+-------------+---------------+---------------+--------------+----------------+-----------+-------+
|Bolt Action |12 GA |N       |12 GA         |0          |0             |0               |0             |0            |0              |0              |1             |0               |0          |1      |
|Bolt Action |16 GA |N       |16 GA         |0          |0             |0               |0             |0            |0              |0              |1             |0               |0     

## Full Pipeline: build_feature_set()

In [8]:
# Run the full pipeline (chains all steps + adds cross-product features + saves to data/features/)
df_features = build_feature_set(df)

print(f"Rows: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")
print(f"\nAll columns:\n{df_features.columns}")
df_features.show(5, truncate=False)

2026-02-26 16:55:20,746 INFO Added time features: month, quarter, year, cyclical, domain events


2026-02-26 16:55:23,116 INFO Added hunting season flags: ['is_waterfowl_season', 'is_upland_season', 'is_turkey_spring_season', 'is_turkey_fall_season', 'is_dove_season', 'is_deer_season'] + hunting_intensity


2026-02-26 16:55:24,858 INFO Added lag features: lags=[1, 3, 6, 12], target=quantity, + avg_price_lag_1, yoy/mom change


2026-02-26 16:55:27,239 INFO Added rolling features: windows=[3, 6, 12], target=quantity


2026-02-26 16:55:31,186 INFO Added product encodings: sizing_grouped, tactical flags, subcategory one-hot (reference=Pump Action, 8 dummies), is_12ga


26/02/26 16:55:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


2026-02-26 16:56:19,902 INFO Saved feature set to /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final/data/features


Rows: 4,440


Columns: 54

All columns:
['subcategory', 'sizing', 'tactical', 'date', 'quantity', 'amount', 'barrel_length', 'transaction_count', 'avg_price', 'month', 'quarter', 'year', 'month_sin', 'month_cos', 'covid_period', 'post_covid', 'is_march', 'off_season', 'months_to_hunting', 'is_waterfowl_season', 'is_upland_season', 'is_turkey_spring_season', 'is_turkey_fall_season', 'is_dove_season', 'is_deer_season', 'hunting_intensity', 'quantity_lag_1', 'quantity_lag_3', 'quantity_lag_6', 'quantity_lag_12', 'avg_price_lag_1', 'quantity_yoy_change', 'quantity_mom_change', 'quantity_ma_3', 'quantity_std_3', 'quantity_ma_6', 'quantity_std_6', 'quantity_ma_12', 'quantity_std_12', 'sizing_grouped', 'is_tactical', 'is_tactical_na', 'is_semiautomatic', 'is_single_shot', 'is_over_under', 'is_lever_action', 'is_side_by_side', 'is_bolt_action', 'is_rifle_shotgun', 'is_revolver', 'is_12ga', 'subcat_total_qty', 'subcat_share', 'time_index']


+-----------+------+--------+----------+--------+---------------+-------------+-----------------+-----------------+-----+-------+----+-------------------+---------------------+------------+----------+--------+----------+-----------------+-------------------+----------------+-----------------------+---------------------+--------------+--------------+-----------------+--------------+--------------+--------------+---------------+-----------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------+-----------+--------------+----------------+--------------+-------------+---------------+---------------+--------------+----------------+-----------+-------+----------------+--------------------+----------+
|subcategory|sizing|tactical|date      |quantity|amount         |barrel_length|transaction_count|avg_price        |month|quarter|year|month_sin          |month_cos            |

## Convert to Pandas

In [9]:
# Convert to pandas (drops warm-up rows with NaN in lag columns)
pdf = to_pandas(df_features)

print(f"Shape: {pdf.shape}")
print(f"\nDtypes:\n{pdf.dtypes}")
print(f"\nNaN counts per column:")
nan_counts = pdf.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols) == 0:
    print("  No NaN values — clean dataset")
else:
    print(nan_cols)

pdf.head()

2026-02-26 16:56:49,370 INFO Converted to pandas: shape=(4440, 54)


2026-02-26 16:56:49,463 INFO Dropped 888 warm-up rows (NaN in lag columns)


2026-02-26 16:56:49,482 INFO Final pandas shape: (3552, 54)


Shape: (3552, 54)

Dtypes:
subcategory                    str
sizing                         str
tactical                       str
date                        object
quantity                     int64
amount                     float64
barrel_length              float64
transaction_count            int64
avg_price                  float64
month                        int32
quarter                      int32
year                         int32
month_sin                  float64
month_cos                  float64
covid_period                 int32
post_covid                   int32
is_march                     int32
off_season                   int32
months_to_hunting            int32
is_waterfowl_season          int32
is_upland_season             int32
is_turkey_spring_season      int32
is_turkey_fall_season        int32
is_dove_season               int32
is_deer_season               int32
hunting_intensity            int32
quantity_lag_1             float64
quantity_lag_3             f

  No NaN values — clean dataset


,subcategory,sizing,tactical,date,quantity,amount,barrel_length,transaction_count,avg_price,month,...,is_over_under,is_lever_action,is_side_by_side,is_bolt_action,is_rifle_shotgun,is_revolver,is_12ga,subcat_total_qty,subcat_share,time_index
12,Bolt Action,12 GA,N,2020-01-01,33,17984.870117,22.0,2,544.996064,1,...,0,0,0,1,0,0,1,439,0.075171,12
13,Bolt Action,12 GA,N,2020-02-01,36,19728.139648,22.0,2,548.003879,2,...,0,0,0,1,0,0,1,501,0.071856,13
14,Bolt Action,12 GA,N,2020-03-01,42,22556.939941,22.0,2,537.069999,3,...,0,0,0,1,0,0,1,315,0.133333,14
15,Bolt Action,12 GA,N,2020-04-01,17,9054.530029,22.0,2,532.619413,4,...,0,0,0,1,0,0,1,216,0.078704,15
16,Bolt Action,12 GA,N,2020-05-01,25,13377.010010,22.0,2,535.080400,5,...,0,0,0,1,0,0,1,276,0.090580,16


## Save Features

In [10]:
# build_feature_set() already saves to data/features/ — verify it exists
features_path = PROJECT_ROOT / "data" / "features"
parquet_files = list(features_path.glob("*.parquet"))
print(f"Features directory: {features_path}")
print(f"Parquet files: {len(parquet_files)}")
for f in parquet_files:
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

Features directory: /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final/data/features
Parquet files: 1
  part-00000-fc7d9c52-8fca-41fb-a8e3-7725c4f13ead-c000.snappy.parquet (326 KB)


In [11]:
spark.stop()
print("SparkSession stopped. Notebook complete.")

SparkSession stopped. Notebook complete.
